In [ ]:
#Importing libraries

!pip install requests beautifulsoup4 --quiet  # The ! means "run this as a terminal command"

import requests          # Makes HTTP requests — like httr in R
from bs4 import BeautifulSoup  # Parses HTML — extracts the data you want from raw webpage code
import pandas as pd      # DataFrames — same concept as data.frame in R
import time              # We'll use time.sleep() to pause between requests (be polite to NECTA's server)
import re                # Regular expressions — for pattern matching in text
import os                # For creating folders to save files in

print("All imports successful")

All imports successful


In [ ]:
BASE_URL = "https://matokeo.necta.go.tz/results/2025/csee"

def get_school_list():
    import string
    all_schools = []

    for letter in string.ascii_lowercase:
        url = f"{BASE_URL}/indexfiles/index_{letter}.htm"
        print(f"Fetching: {url}")

        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
        except Exception as e:
            print(f"  Skipping {letter}: {e}")
            time.sleep(1)
            continue

        soup = BeautifulSoup(response.text, 'html.parser')

        for link in soup.find_all('a', href=True):
            href = link['href']
            text = link.get_text(strip=True)

            # Normalize backslashes to forward slashes
            href_normalized = href.replace('\\', '/')

            # Only S-code school pages (skip P-code center pages, these are primary schools)
            if 'results/s' in href_normalized.lower() and href_normalized.endswith('.htm'):

                # Text format is: "SCHOOL NAME - S5191"
                code_match = re.search(r'-\s*(S\d+)$', text, re.IGNORECASE)
                if code_match:
                    school_code = code_match.group(1).upper()
                    # Name is everything before the " - S####" at the end
                    school_name = re.sub(r'\s*-\s*S\d+$', '', text, flags=re.IGNORECASE).strip()

                    # Build full URL from the normalized relative path
                    # href looks like: ../results/s5191.htm
                    # Base is: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/
                    # So ../results/ resolves to: https://matokeo.necta.go.tz/results/2025/csee/results/
                    filename = href_normalized.split('/')[-1]  # e.g. s5191.htm
                    full_url = f"{BASE_URL}/results/{filename}"

                    all_schools.append({
                        'school_code': school_code,
                        'school_name': school_name,
                        'url': full_url
                    })

        time.sleep(1)

    schools_df = pd.DataFrame(all_schools).drop_duplicates(subset='school_code')
    print(f"\nTotal schools found: {len(schools_df)}")
    return schools_df

schools_df = get_school_list()
schools_df.to_csv('schools.csv', index=False)
print(schools_df[['school_code', 'school_name', 'url']].head(5).to_string())

Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_a.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_b.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_c.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_d.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_e.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_f.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_g.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_h.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_i.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_j.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_k.htm
Fetching: https://matokeo.necta.go.tz/results/2025/csee/indexfiles/index_l.htm
Fetching: https://matokeo.necta.go.tz/results/2025/c

In [ ]:
def parse_school_page(html_text, school_code, school_name):
    soup = BeautifulSoup(html_text, 'html.parser')

    result = {
        'school_code': school_code,
        'school_name': school_name,
        'region': None,
        'gpa': None,
        'registered': None,
        'absent': None,
        'sat': None,
        'total_students': None,
        'div_1': None, 'div_2': None, 'div_3': None, 'div_4': None, 'div_0': None,
        'f_div_1': None, 'f_div_2': None, 'f_div_3': None, 'f_div_4': None, 'f_div_0': None,
        'f_total': None,
        'm_div_1': None, 'm_div_2': None, 'm_div_3': None, 'm_div_4': None, 'm_div_0': None,
        'm_total': None,
    }

    def safe_int(val):
        val = val.strip()
        return int(val) if val.isdigit() else 0

    def safe_float(val):
        try:
            return float(val.strip())
        except:
            return None

    tables = soup.find_all('table')
    if not tables:
        return result

    # --- Table 1: Division summary (F / M / T rows) ---
    rows = tables[0].find_all('tr')
    for row in rows:
        cells = [td.get_text(strip=True) for td in row.find_all(['td', 'th'])]
        if len(cells) < 6 or cells[0] == 'SEX':
            continue
        if cells[0] == 'F':
            result['f_div_1'] = safe_int(cells[1])
            result['f_div_2'] = safe_int(cells[2])
            result['f_div_3'] = safe_int(cells[3])
            result['f_div_4'] = safe_int(cells[4])
            result['f_div_0'] = safe_int(cells[5])
            result['f_total'] = sum(safe_int(cells[i]) for i in range(1, 6))
        elif cells[0] == 'M':
            result['m_div_1'] = safe_int(cells[1])
            result['m_div_2'] = safe_int(cells[2])
            result['m_div_3'] = safe_int(cells[3])
            result['m_div_4'] = safe_int(cells[4])
            result['m_div_0'] = safe_int(cells[5])
            result['m_total'] = sum(safe_int(cells[i]) for i in range(1, 6))
        elif cells[0] == 'T':
            result['div_1'] = safe_int(cells[1])
            result['div_2'] = safe_int(cells[2])
            result['div_3'] = safe_int(cells[3])
            result['div_4'] = safe_int(cells[4])
            result['div_0'] = safe_int(cells[5])
            result['total_students'] = sum(safe_int(cells[i]) for i in range(1, 6))

    # --- Scan page text for region, GPA, enrollment ---
    full_text = soup.get_text(separator=' ')

    # Region
    region_match = re.search(r'EXAMINATION CENTRE REGION\s*[|\s]+([A-Z /]+?)(?:\s*\||\s{2,}|TOTAL)', full_text)
    if region_match:
        result['region'] = region_match.group(1).strip()

    # GPA
    gpa_match = re.search(r'EXAMINATION CENTRE GPA\s*[|\s]+([\d.]+)', full_text)
    if gpa_match:
        result['gpa'] = safe_float(gpa_match.group(1))

    # Enrollment — find the header row, then grab the numbers that follow
    # The page has: REGIST | ABSENT | SAT | WITHHELD | NO-CA | CLEAN | DIV I | ...
    # Followed by:  121    | 6      | 115 | 0        | 0     | 115   | 1     | ...
    enroll_match = re.search(
        r'REGIST\s+ABSENT\s+SAT\s+WITHHELD\s+NO-CA\s+CLEAN\s+DIV I\s+DIV II\s+DIV III\s+DIV IV\s+DIV 0\s+'
        r'(\d+)\s+(\d+)\s+(\d+)',
        full_text
    )
    if enroll_match:
        result['registered'] = int(enroll_match.group(1))
        result['absent']     = int(enroll_match.group(2))
        result['sat']        = int(enroll_match.group(3))

    return result

# Re-run the test
test_url = "https://matokeo.necta.go.tz/results/2025/csee/results/s2734.htm"
test_response = requests.get(test_url, timeout=30)
test_result = parse_school_page(test_response.text, 'S2734', 'NTYUKA SECONDARY SCHOOL')

for key, value in test_result.items():
    print(f"  {key}: {value}")

  school_code: S2734
  school_name: NTYUKA SECONDARY SCHOOL
  region: DODOMA
  gpa: 4.1196
  registered: 121
  absent: 6
  sat: 115
  total_students: 115
  div_1: 1
  div_2: 4
  div_3: 13
  div_4: 75
  div_0: 22
  f_div_1: 0
  f_div_2: 3
  f_div_3: 11
  f_div_4: 45
  f_div_0: 12
  f_total: 71
  m_div_1: 1
  m_div_2: 1
  m_div_3: 2
  m_div_4: 30
  m_div_0: 10
  m_total: 44


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All output goes to Drive — survives runtime resets
DRIVE_BASE = '/content/drive/MyDrive/necta_scrape'
CACHE_DIR = f'{DRIVE_BASE}/raw_html_schools'
os.makedirs(CACHE_DIR, exist_ok=True)

def scrape_all_schools(schools_df, delay_seconds=1.5):
    all_results = []
    total = len(schools_df)

    for idx, row in schools_df.iterrows():
        school_code = row['school_code']
        school_name = row['school_name']
        url = row['url']

        if idx % 100 == 0:
            print(f"Progress: {idx}/{total} schools scraped...")

        cache_path = f"{CACHE_DIR}/{school_code.lower()}.html"

        if os.path.exists(cache_path):
            with open(cache_path, 'r', encoding='utf-8') as f:
                html_text = f.read()
        else:
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                html_text = response.text
                with open(cache_path, 'w', encoding='utf-8') as f:
                    f.write(html_text)
                time.sleep(delay_seconds)
            except Exception as e:
                print(f"  ERROR on {school_code} ({url}): {e}")
                all_results.append({'school_code': school_code, 'school_name': school_name, 'error': str(e)})
                continue

        parsed = parse_school_page(html_text, school_code, school_name)
        all_results.append(parsed)

        # Save checkpoint to Drive every 200 schools
        # This way if it resets at school 3000, you don't lose everything
        if idx % 200 == 0 and idx > 0:
            checkpoint_df = pd.DataFrame(all_results)
            checkpoint_df.to_csv(f'{DRIVE_BASE}/checkpoint_{idx}.csv', index=False)
            print(f"  Checkpoint saved at {idx} schools")

    print(f"\nDone. Scraped {len(all_results)} schools.")
    final_df = pd.DataFrame(all_results)
    final_df.to_csv(f'{DRIVE_BASE}/necta_2025_schools.csv', index=False)
    print(f"Saved to {DRIVE_BASE}/necta_2025_schools.csv")
    return final_df

results_df = scrape_all_schools(schools_df)
print(results_df.shape)
print(results_df['region'].value_counts().head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Progress: 0/5864 schools scraped...
Progress: 100/5864 schools scraped...
Progress: 200/5864 schools scraped...
  Checkpoint saved at 200 schools
Progress: 300/5864 schools scraped...
Progress: 400/5864 schools scraped...
  Checkpoint saved at 400 schools
Progress: 500/5864 schools scraped...
Progress: 600/5864 schools scraped...
  Checkpoint saved at 600 schools
Progress: 700/5864 schools scraped...
Progress: 800/5864 schools scraped...
  Checkpoint saved at 800 schools
Progress: 900/5864 schools scraped...
Progress: 1000/5864 schools scraped...
  Checkpoint saved at 1000 schools
Progress: 1100/5864 schools scraped...
Progress: 1200/5864 schools scraped...
  Checkpoint saved at 1200 schools
Progress: 1300/5864 schools scraped...
Progress: 1400/5864 schools scraped...
  Checkpoint saved at 1400 schools
  ERROR on S2797 (https://matokeo.necta.go.tz/results/202

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('necta_2025_schools.csv', '/content/drive/MyDrive/necta_2025_schools.csv')
print("Saved to Google Drive")

In [ ]:
import os

# Check what files actually exist in the current directory
print("Files in current directory:")
print(os.listdir('.'))

# Check if the raw HTML cache exists
if os.path.exists('raw_html_schools'):
    html_files = os.listdir('raw_html_schools')
    print(f"\nCached HTML files: {len(html_files)}")
    print("First 5:", html_files[:5])
else:
    print("\nNo raw_html_schools folder found")